# Multivariate time series forecasting using  LSTM

Predicts a future value (e.g., temperature) using multiple (3) input features (e.g., temperature, humidity, pressure, etc.).

In [ ]:
import numpy as np
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

# -------------------------------
# 1. Create synthetic multivariate data
# -------------------------------
np.random.seed(42)
timesteps = 300
t = np.arange(timesteps)

# Example features: temperature, humidity, pressure
temperature = 20 + 0.05 * t + np.random.normal(0, 0.5, timesteps)
humidity = 60 + np.sin(0.1 * t) * 10 + np.random.normal(0, 1, timesteps)
pressure = 1013 + np.cos(0.1 * t) * 5 + np.random.normal(0, 0.5, timesteps)

data = pd.DataFrame({
    'temperature': temperature,
    'humidity': humidity,
    'pressure': pressure
})

print("Data shape:", data.shape)
print(data.head(20))

In [ ]:
# -------------------------------
# 2. Prepare data for supervised learning
# -------------------------------
def create_dataset(dataset, look_back=3):
    X, y = [], []
    for i in range(len(dataset) - look_back):
        X.append(dataset[i:(i + look_back), :])
        y.append(dataset[i + look_back, 0])  # Predict temperature
    return np.array(X), np.array(y)

# Normalize all features
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(data)
print("Scaled data shape:", scaled_data.shape)
print(scaled_data[:20])

In [ ]:
look_back = 7
X, y = create_dataset(scaled_data, look_back)

print("--- X shape:", X.shape)  # (samples, timesteps, features)
print(X[:5])
print("--- y shape:", y.shape)
print(y[:10])

In [ ]:
# Split into train and test using sklearn.model_selection
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
print("X_train shape:", X_train.shape)
print(f"-----------------------------------X_train:\n{X_train[:10]}")
print("y_train shape:", y_train.shape)
print(f"-----------------------------------y_train:\n{y_train[:10]}")
print("X_test shape:", X_test.shape)
print(f"-----------------------------------X_test:\n{X_test[:10]}")
print("y_test shape:", y_test.shape)
print(f"-----------------------------------y_test:\n{y_test[:10]}")

In [ ]:
model = Sequential([
    LSTM(64, input_shape=(X_train.shape[1], X_train.shape[2]), return_sequences=True),
    LSTM(32, activation='relu'),
    Dense(32, activation='relu'),
    Dense(1)
])

model.compile(optimizer='adam', loss='mse')

history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=16,
    validation_data=(X_test, y_test),
    verbose=0
)

import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error

predicted = model.predict(X_test)

dummy_predicted = np.zeros((len(predicted), scaled_data.shape[1]))
dummy_predicted[:, 0] = predicted[:, 0]
predicted_unscaled = scaler.inverse_transform(dummy_predicted)[:, 0]

dummy_y_test = np.zeros((len(y_test), scaled_data.shape[1]))
dummy_y_test[:, 0] = y_test
y_test_unscaled = scaler.inverse_transform(dummy_y_test)[:, 0]


plt.figure(figsize=(10, 6))
plt.plot(y_test_unscaled, label='Actual (Unscaled)')
plt.plot(predicted_unscaled, label='Predicted (Unscaled)')
plt.title('Temperature Forecasting (LSTM) - Unscaled')
plt.xlabel('Time Steps')
plt.ylabel('Temperature')
plt.legend()
plt.show()

plt.figure(figsize=(10, 6))
plt.plot(y_test, label='Actual (Scaled)')
plt.plot(predicted, label='Predicted (Scaled)')
plt.title('Temperature Forecasting (LSTM) - Scaled')
plt.xlabel('Time Steps')
plt.ylabel('Scaled Temperature')
plt.legend()
plt.show()

rmse_unscaled = np.sqrt(mean_squared_error(y_test_unscaled, predicted_unscaled))
print(f"Root Mean Squared Error (RMSE) on unscaled test data: {rmse_unscaled:.4f}")

rmse_scaled = np.sqrt(mean_squared_error(y_test, predicted))
print(f"Root Mean Squared Error (RMSE) on scaled test data: {rmse_scaled:.4f}")